In [ ]:
import jax
import numpy as np
import jax.numpy as jnp

from tensordev.core.utils.shuffle_precalculation import assemble_shuffle_algebra
from tensordev.core.utils.shuffle_product import shuffle_product

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
# --- ASSEMBLE SHUFFLE ALGEBRA ---
# Set dimension and truncation level
d = 2
N = 2

# Assemble (precompute) shuffle algebra
# Attention: Use Jax version
shuffle_algebra = assemble_shuffle_algebra(d, N)

# Set up example tensors (note batch dimension of 2)
A = (np.array([[1.0],[1.0]]), np.array([[0.0,1.0],[1.0,0.0]]), np.array([[2.0,3.0,0.0,0.0], [0.0,0.0,0.0,0.0]]))
B = (np.array([[2.0],[2.0]]), np.array([[2.0,0.0],[3.0,0.0]]), np.array([[1.0,0.0,0.0,1.0], [0.0,0.0,0.0,0.0]]))
A

(array([[1.],
        [1.]]),
 array([[0., 1.],
        [1., 0.]]),
 array([[2., 3., 0., 0.],
        [0., 0., 0., 0.]]))

In [45]:
# --- COMPUTE SHUFFLE PRODUCT ---
# Attention: Use Jax version
res = shuffle_product(A, B, shuffle_algebra)
res # Correct shuffle product of A and B

(Array([[2.],
        [2.]], dtype=float32),
 Array([[2., 2.],
        [5., 0.]], dtype=float32),
 Array([[5., 8., 2., 1.],
        [6., 0., 0., 0.]], dtype=float32),
 Array([[12., 13.,  7.,  0.,  1.,  0.,  0.,  3.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.]], dtype=float32),
 Array([[12.,  9.,  6.,  2.,  3.,  2.,  2.,  9.,  0.,  2.,  2.,  6.,  2.,
          3.,  0.,  0.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,
          0.,  0.,  0.]], dtype=float32))

In [47]:
# --- COMPUTE GRADIENT INVOLVING SHUFFLE PRODUCT ---
# This example shows how to generically calculate the gradient of a scalar function involing the shuffle product

# Set up scalar function f(A,B) = mean_over_batch(sum((A \shuffle B)^2))
def loss_fn(A: tuple, B: tuple, shuffle_algebra: dict):
    # Perform shuffle product
    C = shuffle_product(A, B, shuffle_algebra)

    # Evaluate function over batch dimension
    batch_total_sq = sum([jnp.sum(jnp.square(t_hom), axis=1) for t_hom in C])
    return jnp.sum(batch_total_sq) / A[0].shape[0] # Divide by number of batches

# See result 
print(f"{loss_fn(A, B, shuffle_algebra) = }")

# Create Gradient function (argnums=(0,1) tells JAX to differentiate w.r.t. A and B)
grad_fn = jax.grad(loss_fn, argnums=(0, 1))

# Calculate actual gradient 
grad_A, grad_B = grad_fn(A, B, shuffle_algebra)
grad_A

loss_fn(A, B, shuffle_algebra) = Array(481.5, dtype=float32)


(Array([[14.],
        [19.]], dtype=float32),
 Array([[60., 54.],
        [46.,  0.]], dtype=float32),
 Array([[166., 166.,  46.,  14.],
        [ 12.,   0.,   0.,   0.]], dtype=float32))